In [1]:
%pwd

'd:\\Tipto\\OmniChef-Nexus\\notebooks'

In [2]:
import os 
os.chdir("../")

In [3]:
%pwd

'd:\\Tipto\\OmniChef-Nexus'

In [4]:
import ast
import json
import io
from pathlib import Path

import pandas as pd
from PIL import Image
from datasets import Dataset, Features, Value, Image as HFImage, Sequence

In [5]:
df = pd.read_csv("data/all csv files/recipes_15k_samples.csv")
df.shape

(15698, 15)

In [6]:
print(f"Loaded {len(df)} rows | columns: {df.columns.tolist()}")

Loaded 15698 rows | columns: ['name', 'minutes', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients', 'recipe_id', 'rating', 'review', 'num of ratings', 'markdown_recipe', 'image_path']


In [7]:
df.sample(1)

,name,minutes,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,recipe_id,rating,review,num of ratings,markdown_recipe,image_path
10670,oven roasted sweet potato fries,40,"['60-minutes-or-less', 'time-to-make', 'course...","[154.7, 3.0, 25.0, 3.0, 5.0, 1.0, 10.0]",8,"['preheat the oven to 425', 'wash and clean po...","healthful, spicy, side dish that even some peo...","['sweet potatoes', 'olive oil', 'paprika', 'gr...",5,65112,4.538462,"[""My friends loved this to snack on while play...",13.0,# Oven Roasted Sweet Potato Fries\n**Recipe ID...,data/output/images/recipe_id_65112.png


In [8]:
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
CSV_PATH = "data/all csv files/recipes_15k_samples.csv" 
IMAGE_ROOT = Path("data/output/images")
REPO_ID = "tiptoghosh/food-recipes-15k"
HF_TOKEN = os.getenv("HF_TOKEN")

In [10]:
# Nutrition list positional mapping
NUTRITION_KEYS = [
    "calories", "total_fat_pdv", "sugar_pdv",
    "sodium_pdv", "protein_pdv", "saturated_fat_pdv", "carbs_pdv"
]

In [11]:
def parse_list_field(val) -> list:
    """Safely parse a stringified Python list into an actual list."""
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            result = ast.literal_eval(val)
            return result if isinstance(result, list) else [val]
        except Exception:
            return [val]
    return []

In [12]:
def parse_nutrition(val) -> dict:
    """
    Convert nutrition list [calories, total_fat, sugar, sodium,
    protein, saturated_fat, carbs] → labelled dict of floats.
    Returns empty dict if malformed.
    """
    raw = parse_list_field(val)
    if len(raw) != 7:
        return {}
    try:
        return {k: float(v) for k, v in zip(NUTRITION_KEYS, raw)}
    except (ValueError, TypeError):
        return {}

In [13]:
def load_image_bytes(image_path_str: str) -> bytes | None:
    """
    Load image from path → JPEG bytes.
    Returns None if file is missing or unreadable.
    """
    if not image_path_str:
        return None
    p = Path(image_path_str)
    # Try as-is first (absolute/relative), then relative to IMAGE_ROOT
    candidates = [p, IMAGE_ROOT / p.name]
    for candidate in candidates:
        if candidate.exists():
            try:
                img = Image.open(candidate).convert("RGB")
                buf = io.BytesIO()
                img.save(buf, format="JPEG", quality=90)
                return buf.getvalue()
            except Exception:
                return None
    return None

In [14]:
sample_nutrition = parse_nutrition(df["nutrition"].iloc[0])
sample_nutrition

{'calories': 105.7,
 'total_fat_pdv': 8.0,
 'sugar_pdv': 0.0,
 'sodium_pdv': 26.0,
 'protein_pdv': 5.0,
 'saturated_fat_pdv': 4.0,
 'carbs_pdv': 3.0}

In [15]:
sample_steps = parse_list_field(df["steps"].iloc[0])
sample_steps

['dredge pork chops in mixture of flour , salt , dry mustard and garlic powder',
 'brown in oil in a large skillet',
 'place browned pork chops in a crock pot',
 'add the can of soup , undiluted',
 'cover and cook on low for 6-8 hours']

In [16]:
def recipe_generator():
    missing_images = 0
    malformed_nutrition = 0

    for idx, row in df.iterrows():
        # Parse list fields 
        steps       = parse_list_field(row.get("steps", []))
        ingredients = parse_list_field(row.get("ingredients", []))
        tags        = parse_list_field(row.get("tags", []))
        nutrition   = parse_nutrition(row.get("nutrition", []))

        if not nutrition:
            malformed_nutrition += 1

        # Safe scalar fields
        name  = row["name"] if pd.notna(row.get("name")) else ""
        description  = row["description"] if pd.notna(row.get("description")) else ""
        markdown = row["markdown_recipe"] if pd.notna(row.get("markdown_recipe")) else ""
        recipe_id = str(row.get("recipe_id", "")).strip()
        minutes = int(row["minutes"]) if pd.notna(row.get("minutes")) else -1
        n_steps = int(row["n_steps"]) if pd.notna(row.get("n_steps")) else len(steps)
        n_ingredients = int(row["n_ingredients"]) if pd.notna(row.get("n_ingredients")) else len(ingredients)
        rating  = float(row["rating"]) if pd.notna(row.get("rating")) else -1.0
        num_ratings = float(row["num of ratings"]) if pd.notna(row.get("num of ratings")) else 0.0

        # Image (one at a time — no RAM buildup)
        img_bytes = load_image_bytes(str(row.get("image_path", "")))
        if img_bytes is None:
            missing_images += 1

        # Progress log every 1000 rows
        if idx % 1000 == 0:
            print(f"  [{idx:>5}] missing_images={missing_images} | bad_nutrition={malformed_nutrition}")

        yield {
            "name": name,
            "recipe_id": recipe_id,
            "minutes": minutes,
            "description": description,
            "tags": tags,
            "steps": steps,
            "n_steps": n_steps,
            "ingredients": ingredients,
            "n_ingredients": n_ingredients,
            "nutrition": nutrition,  
            "rating": rating,
            "num_ratings": num_ratings,
            "markdown":  markdown,
            "image": img_bytes,   
        }
        
    print(f"\nGenerator complete — missing images: {missing_images} | malformed nutrition: {malformed_nutrition}")

In [17]:
print(df.dtypes)

name                   str
minutes              int64
tags                   str
nutrition              str
n_steps              int64
steps                  str
description            str
ingredients            str
n_ingredients        int64
recipe_id            int64
rating             float64
review                 str
num of ratings     float64
markdown_recipe        str
image_path             str
dtype: object


In [18]:
# Defining this upfront prevents HF from inferring wrong types
features = Features({
    "name": Value("string"),
    "recipe_id": Value("int64"),
    "minutes":       Value("int64"),
    "description":   Value("string"),
    "tags":          Sequence(Value("string")),
    "steps":         Sequence(Value("string")),
    "n_steps":       Value("int64"),
    "n_ingredients": Value("int64"),
    "n_ingredients": Value("int32"),
    "nutrition":     {                         
        "calories":         Value("float32"),
        "total_fat_pdv":    Value("float32"),
        "sugar_pdv":        Value("float32"),
        "sodium_pdv":       Value("float32"),
        "protein_pdv":      Value("float32"),
        "saturated_fat_pdv":Value("float32"),
        "carbs_pdv":        Value("float32"),
    },
    "rating":        Value("float32"),
    "num_ratings":   Value("float32"),
    "markdown":      Value("string"),
    "image":         HFImage(),                
})

In [19]:
dataset = Dataset.from_generator(
    recipe_generator , features = features
)

Loading dataset shards:   0%|          | 0/27 [00:00<?, ?it/s]

In [20]:
dataset

Dataset({
    features: ['name', 'recipe_id', 'minutes', 'description', 'tags', 'steps', 'n_steps', 'n_ingredients', 'nutrition', 'rating', 'num_ratings', 'markdown', 'image'],
    num_rows: 15698
})

In [21]:
from huggingface_hub import HfApi

api = HfApi(token = HF_TOKEN)

api.delete_repo(
    repo_id=REPO_ID,
    repo_type="dataset",
    missing_ok=True      
)

In [22]:
api.create_repo(
    repo_id=REPO_ID,
    repo_type="dataset",
    private=False
)

RepoUrl('https://huggingface.co/datasets/tiptoghosh/food-recipes-15k', endpoint='https://huggingface.co', repo_type='dataset', repo_id='tiptoghosh/food-recipes-15k')

In [23]:
dataset.push_to_hub(
    repo_id = REPO_ID,
    token = HF_TOKEN,
    private = False,
    max_shard_size = "500MB",
)

Uploading the dataset shards:   0%|          | 0/32 [00:00<?, ? shards/s]

Map:   0%|          | 0/491 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Map:   0%|          | 0/490 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/tiptoghosh/food-recipes-15k/commit/72d22ecf360f6e39ed6c5e223bff3085d1fd84b6', commit_message='Upload dataset', commit_description='', oid='72d22ecf360f6e39ed6c5e223bff3085d1fd84b6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/tiptoghosh/food-recipes-15k', endpoint='https://huggingface.co', repo_type='dataset', repo_id='tiptoghosh/food-recipes-15k'), pr_revision=None, pr_num=None)